In [2]:
import json
import os

FHIR_DIR = "../synthea/output/fhir"

def investigate_missing_drugs(limit=3):
    """Scans FHIR bundles for MedicationRequests that lack standard coding to see where the data hides."""
    print("🕵️ STARTING DATA DETECTIVE MODE...\n" + "-"*50)

    json_files = [f for f in os.listdir(FHIR_DIR) if f.endswith('.json')]
    found_cases = 0

    for file in json_files:
        if found_cases >= limit:
            break

        file_path = os.path.join(FHIR_DIR, file)
        with open(file_path, encoding='utf-8') as f:
            fhir_data = json.load(f)

        for entry in fhir_data.get('entry', []):
            resource = entry.get('resource', {})

            # Look for Prescriptions
            if resource.get('resourceType') == 'MedicationRequest':
                medication = resource.get('medicationCodeableConcept', {})
                coding = medication.get('coding', [])

                # If the standard coding list is empty (which causes our 'Unknown')
                if not coding:
                    print(f"📄 File: {file}")
                    print("🔍 Raw JSON block for the missing drug:")

                    # Extract only the keys relevant to the medication to keep the output readable
                    med_data = {k: v for k, v in resource.items() if 'medication' in k.lower()}
                    print(json.dumps(med_data, indent=2))
                    print("-" * 50)

                    found_cases += 1
                    if found_cases >= limit:
                        break

    if found_cases == 0:
         print("✅ Searched all files. Could not find any empty MedicationRequests. The issue might be elsewhere.")

investigate_missing_drugs(limit=3)

🕵️ STARTING DATA DETECTIVE MODE...
--------------------------------------------------
📄 File: Ahmed109_VonRueden376_0190fb66-1ec0-482d-02cb-f08d3defbe04.json
🔍 Raw JSON block for the missing drug:
{
  "medicationReference": {
    "reference": "urn:uuid:17e0464c-0610-f406-2fe1-d7c7047ff9fa"
  }
}
--------------------------------------------------
📄 File: Ahmed109_VonRueden376_0190fb66-1ec0-482d-02cb-f08d3defbe04.json
🔍 Raw JSON block for the missing drug:
{
  "medicationReference": {
    "reference": "urn:uuid:76e9aa73-3157-ec36-27c4-bbf268daf3b5"
  }
}
--------------------------------------------------
📄 File: Ahmed109_VonRueden376_0190fb66-1ec0-482d-02cb-f08d3defbe04.json
🔍 Raw JSON block for the missing drug:
{
  "medicationReference": {
    "reference": "urn:uuid:46f2b6de-4326-3f5b-9ee6-2f0fc18da5d1"
  }
}
--------------------------------------------------
